# Notebook 13 — Missing Values

## Learning objectives
Detect, remove, fill, or interpolate missing data and choose a strategy based on context.

**How to use this notebook:** Read each Markdown cell, run the following Python cell, compare your output with the explanation, and then try changing the sample values. The examples use small, fictional datasets so they can run locally without downloading data.

## Setup
NumPy and Pandas are common Python libraries for numerical computation and tabular data analysis. If they are not installed, run `pip install numpy pandas openpyxl pyarrow` in your terminal or notebook environment.

In [1]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.3.5
Pandas version: 2.2.3


## Sample dataset

In [2]:
df = pd.DataFrame({
    'age': [25, np.nan, 35, 40, np.nan],
    'income': [40000, 52000, np.nan, 61000, 58000],
    'score': [70.0, 72.0, np.nan, 80.0, 82.0]
})
print(df)

    age   income  score
0  25.0  40000.0   70.0
1   NaN  52000.0   72.0
2  35.0      NaN    NaN
3  40.0  61000.0   80.0
4   NaN  58000.0   82.0


## isnull() and notnull()

### 1. Concept explanation
These methods identify missing and present values.

### 2. Real-world analogy
Mark empty answers on a form.

### 3. Business example
Measure data completeness by column.

### 4. AI/ML example
Detect missing feature values before training.

In [3]:
print(df.isnull())
print(df.notnull().sum())
print(df.isnull().sum())

     age  income  score
0  False   False  False
1   True   False  False
2  False    True   True
3  False   False  False
4   True   False  False
age       3
income    4
score     4
dtype: int64
age       2
income    1
score     1
dtype: int64


### 5. Output explanation
The first output marks missing cells; the counts show how many values are present or missing in each column.

### 6. Best practices
Measure missingness before deciding what to do.

### 7. Common mistakes
A zero, empty string, or special code may not be recognized as missing automatically.

### 8. Advantages
Quick data-quality check.

### 9. Limitations
Detection does not determine the correct treatment.

## dropna()

### 1. Concept explanation
`dropna` removes rows or columns according to missingness rules.

### 2. Real-world analogy
Discard forms that are incomplete for a required field.

### 3. Business example
Remove records missing a critical identifier.

### 4. AI/ML example
Drop unusable training rows only when justified.

In [4]:
print(df.dropna())
print(df.dropna(subset=['age']))

    age   income  score
0  25.0  40000.0   70.0
3  40.0  61000.0   80.0
    age   income  score
0  25.0  40000.0   70.0
2  35.0      NaN    NaN
3  40.0  61000.0   80.0


### 5. Output explanation
The first result keeps only fully complete rows; the second keeps rows with age present.

### 6. Best practices
Use `subset` to specify critical fields and record how many rows are removed.

### 7. Common mistakes
Dropping all incomplete rows can discard too much data.

### 8. Advantages
Simple when missingness is limited.

### 9. Limitations
Can introduce bias if missingness is systematic.

## fillna() and interpolation

### 1. Concept explanation
`fillna` replaces missing values with chosen values or statistics; interpolation estimates values between known observations.

### 2. Real-world analogy
Fill a blank with a documented default or estimate a gap in a trend.

### 3. Business example
Fill numeric fields with a median or time-series gaps with interpolation.

### 4. AI/ML example
Impute features using training data only to avoid leakage.

In [5]:
filled = df.copy()
filled['age'] = filled['age'].fillna(filled['age'].median())
filled['income'] = filled['income'].fillna(filled['income'].median())
series = pd.Series([10.0, np.nan, 30.0])
print(filled)
print(series.interpolate())

    age   income  score
0  25.0  40000.0   70.0
1  35.0  52000.0   72.0
2  35.0  55000.0    NaN
3  40.0  61000.0   80.0
4  35.0  58000.0   82.0
0    10.0
1    20.0
2    30.0
dtype: float64


### 5. Output explanation
Missing ages and incomes are filled with column medians; the middle series value is interpolated to 20.

### 6. Best practices
Fit imputation values on training data and reuse them for validation/test data.

### 7. Common mistakes
Computing medians using the entire dataset before splitting can leak information.

### 8. Advantages
Keeps more records for analysis.

### 9. Limitations
Imputed values are estimates, not observed facts.

## Missing-value handling strategies

### 1. Concept explanation
Common choices include deletion, constant filling, mean/median/mode imputation, group-wise imputation, interpolation, and explicit missing indicators.

### 2. Real-world analogy
Choose a repair method based on why a field is blank.

### 3. Business example
Use median income imputation when justified, or preserve unknown as a category.

### 4. AI/ML example
Use a pipeline with imputation fitted only on training folds.

In [6]:
df2 = df.copy()
df2['age_was_missing'] = df2['age'].isna().astype('int8')
df2['age'] = df2['age'].fillna(df2['age'].median())
print(df2)

    age   income  score  age_was_missing
0  25.0  40000.0   70.0                0
1  35.0  52000.0   72.0                1
2  35.0      NaN    NaN                0
3  40.0  61000.0   80.0                0
4  35.0  58000.0   82.0                1


### 5. Output explanation
A binary indicator records which ages were missing before median filling.

### 6. Best practices
Understand why data is missing and document assumptions.

### 7. Common mistakes
Blindly filling all columns with zero may distort meaning.

### 8. Advantages
Flexible strategies for different data types.

### 9. Limitations
No universal strategy works for every dataset.